# Using LiteLLM with Pydantic AI

This cookbook demonstrates how to use LiteLLM's proxy and Router with [Pydantic AI](https://pydantic-ai.readthedocs.io/) agents.

LiteLLM exposes an OpenAI-compatible endpoint at `/v1/...` which Pydantic AI's OpenAI model provider can use directly. This allows you to:
- Route requests across multiple LLM providers (OpenAI, Anthropic, Google, etc.)
- Add load balancing, fallbacks, and rate limiting
- Use a single API key and endpoint for all models

## Prerequisites

Install the required packages:

In [ ]:
!pip install litellm pydantic-ai

## 1. Start the LiteLLM Proxy

First, create a `config.yaml` for the LiteLLM proxy. This configures multiple providers behind a single OpenAI-compatible endpoint:

In [ ]:
%%writefile litellm_config.yaml
model_list:
  - model_name: gpt-4o
    litellm_params:
      model: openai/gpt-4o
      api_key: os.environ/OPENAI_API_KEY
  - model_name: claude-sonnet
    litellm_params:
      model: anthropic/claude-3-5-sonnet-20241022
      api_key: os.environ/ANTHROPIC_API_KEY
  - model_name: gemini-pro
    litellm_params:
      model: gemini/gemini-2.0-flash
      api_key: os.environ/GEMINI_API_KEY
general_settings:
  master_key: sk-litellm-test  # API key for proxy auth

Start the proxy server (in a separate terminal):

```bash
litellm --config litellm_config.yaml --port 4000
```

## 2. Use Pydantic AI with the LiteLLM Proxy

Once the proxy is running, configure your Pydantic AI agent to point at the proxy's OpenAI-compatible endpoint:

In [ ]:
import os
from pydantic_ai import Agent

# Point Pydantic AI at the LiteLLM proxy
# The proxy speaks the OpenAI API protocol, so we use the 'openai' provider
agent = Agent(
    'openai:gpt-4o',
    base_url='http://localhost:4000/v1',
    api_key='sk-litellm-test',
)

result = agent.run_sync('What is the capital of France?')
print(result.data)

## 2b. (Alternative) Use LiteLLM Router Directly in Python

If you don't want to run a proxy server, you can use LiteLLM's `Router` class directly in your Python code. Pydantic AI can still connect through a lightweight local proxy:

In [ ]:
from litellm import Router

# Configure a Router with multiple providers
model_list = [
    {
        "model_name": "gpt-4o",
        "litellm_params": {
            "model": "openai/gpt-4o",
            "api_key": os.environ.get("OPENAI_API_KEY"),
        },
    },
    {
        "model_name": "claude-sonnet",
        "litellm_params": {
            "model": "anthropic/claude-3-5-sonnet-20241022",
            "api_key": os.environ.get("ANTHROPIC_API_KEY"),
        },
    },
]

router = Router(model_list=model_list)

# Use the router directly with LiteLLM's completion
response = router.completion(
    model="gpt-4o",
    messages=[{"role": "user", "content": "Say hello!"}],
)
print(response['choices'][0]['message']['content'])

## 3. Structured Output with Pydantic AI + LiteLLM Proxy

Pydantic AI's structured result types work seamlessly through the LiteLLM proxy:

In [ ]:
from pydantic import BaseModel


class City(BaseModel):
    name: str
    country: str
    population: int


agent = Agent(
    'openai:gpt-4o',
    base_url='http://localhost:4000/v1',
    api_key='sk-litellm-test',
    result_type=list[City],
    system_prompt='List the 3 largest cities in Europe with their countries and populations.',
)

result = agent.run_sync('Generate the list')
for city in result.data:
    print(f"{city.name}, {city.country} - Population: {city.population:,}")

## 4. Tool Use with Pydantic AI through LiteLLM

Pydantic AI's function tool support works through the proxy:

In [ ]:
from pydantic_ai import RunContext


def get_weather(ctx: RunContext, city: str) -> str:
    return f"The weather in {city} is sunny, 72\u00b0F."


agent = Agent(
    'openai:gpt-4o',
    base_url='http://localhost:4000/v1',
    api_key='sk-litellm-test',
    tools=[get_weather],
)

result = agent.run_sync('What is the weather in Paris?')
print(result.data)

## 5. Switching Between Models

Since the proxy exposes all configured models behind the same endpoint, you can switch models easily:

In [ ]:
# Use Claude via LiteLLM proxy
claude_agent = Agent(
    'openai:claude-sonnet',  # model_name from proxy config
    base_url='http://localhost:4000/v1',
    api_key='sk-litellm-test',
)

result = claude_agent.run_sync('Explain quantum computing in one sentence.')
print(f"[Claude]: {result.data}")

# Use Gemini via LiteLLM proxy
gemini_agent = Agent(
    'openai:gemini-pro',
    base_url='http://localhost:4000/v1',
    api_key='sk-litellm-test',
)

result = gemini_agent.run_sync('Explain quantum computing in one sentence.')
print(f"[Gemini]: {result.data}")

## Summary

- LiteLLM's proxy exposes an OpenAI-compatible `/v1/...` endpoint
- Pydantic AI's `openai:` provider can consume this endpoint by setting `base_url` and `api_key`
- All Pydantic AI features (structured outputs, tools, multi-model) work through the proxy
- The LiteLLM Router is also available for direct Python integration without a proxy server

For more details, see:
- [LiteLLM Proxy Docs](https://docs.litellm.ai/docs/proxy)
- [Pydantic AI Docs](https://pydantic-ai.readthedocs.io/)